# Betting Odds Data — Preprocessing
Clean `ufc_fights_with_betting_odds.csv` and check overlap with `ufc_event_fight_stats.csv`.

In [157]:
import pandas as pd
import numpy as np
from pybettor import implied_prob

## Load Data

In [158]:
odds = pd.read_csv('../data/raw/ufc_fights_with_betting_odds.csv')
# CHANGED: load from fights_cleaned.csv (not raw) so fight_id is available
# fight_stats = pd.read_csv('../data/raw/ufc_event_fight_stats.csv')  # old
fight_stats = pd.read_csv('../data/processed/fights_cleaned.csv')
events = pd.read_csv('../data/raw/ufc_events.csv')

print('odds:      ', odds.shape)
print('fight_stats:', fight_stats.shape)
print('events:    ', events.shape)

odds:       (5528, 11)
fight_stats: (6900, 63)
events:     (711, 4)


In [159]:
odds.head(3)

,R_fighter,B_fighter,R_odds,B_odds,date,location,country,Winner,title_bout,weight_class,gender
0,Kevin Lee,Charles Oliveira,-145.0,135.0,3/14/2020,"Brasilia, Distrito Federal, Brazil",Brazil,Blue,False,Lightweight,MALE
1,Demian Maia,Gilbert Burns,160.0,-170.0,3/14/2020,"Brasilia, Distrito Federal, Brazil",Brazil,Blue,False,Welterweight,MALE
2,Renato Moicano,Damir Hadzovic,-345.0,315.0,3/14/2020,"Brasilia, Distrito Federal, Brazil",Brazil,Red,False,Lightweight,MALE


## Check Overlap With Fight Stats
Both datasets lack a shared ID, so matches are done on normalized fighter names + event date.

**Strategy:**
1. Join `fight_stats` with `events` on `event_url` / `url_link` to attach a date to each fight.
2. Normalize names in both datasets (lowercase, strip whitespace).
3. For each odds row, look for a matching fight_stats row where `{R_fighter, B_fighter}` equals `{f1_name, f2_name}` (order-insensitive) on the same date.

In [160]:
# Attach event date to fight_stats via event_url
events_slim = events[['url_link', 'event_date']].rename(columns={'url_link': 'event_url'})
fight_stats = fight_stats.merge(events_slim, on='event_url', how='left')

print('fight_stats with date — missing dates:', fight_stats['event_date'].isnull().sum())
fight_stats[['f1_name', 'f2_name', 'event_date']].head(3)

fight_stats with date — missing dates: 0


,f1_name,f2_name,event_date
0,Cory Sandhagen,Umar Nurmagomedov,2024-08-03
1,Shara Magomedov,Michal Oleksiejczuk,2024-08-03
2,Jai Herbert,Rolando Bedoya,2024-08-03


In [161]:
# Normalize names: lowercase + strip
def norm(s):
    return s.str.lower().str.strip()

odds['r_norm'] = norm(odds['R_fighter'])
odds['b_norm'] = norm(odds['B_fighter'])
odds['pair'] = odds.apply(lambda r: frozenset([r['r_norm'], r['b_norm']]), axis=1)
odds['date_norm'] = pd.to_datetime(odds['date']).dt.date

fight_stats['f1_norm'] = norm(fight_stats['f1_name'])
fight_stats['f2_norm'] = norm(fight_stats['f2_name'])
fight_stats['pair'] = fight_stats.apply(lambda r: frozenset([r['f1_norm'], r['f2_norm']]), axis=1)
fight_stats['date_norm'] = pd.to_datetime(fight_stats['event_date']).dt.date

In [162]:
# Join fight_id from fight_stats into odds via order-insensitive (pair, date_norm) match
# Also record which odds fighter (R or B) corresponds to f1 in the fight_stats dataset

stats_keys = fight_stats[['f1_name', 'f2_name', 'pair', 'date_norm', 'fight_id']].drop_duplicates()

odds = odds.merge(stats_keys, on=['pair', 'date_norm'], how='left')

# Determine R/B to f1/f2 alignment
# r_is_f1 = 1 means R_fighter is f1 in the fight_stats row, 0 means R_fighter is f2
odds['r_norm_check'] = odds['R_fighter'].str.lower().str.strip()
odds['r_is_f1'] = (odds['r_norm_check'] == odds['f1_name'].str.lower().str.strip()).astype('Int64')

# Drop intermediate alignment columns
odds.drop(columns=['r_norm_check', 'f1_name', 'f2_name'], inplace=True)

before_fid = len(odds)
odds = odds[odds['fight_id'].notna()].copy()
dropped_fid = before_fid - len(odds)

# Convert to nullable integer type after dropping nulls
odds['fight_id'] = odds['fight_id'].astype('Int64')

matched_count = odds['fight_id'].notna().sum()
print(f"Dropped {dropped_fid} rows with no fight_id. Remaining rows with fight_id: {len(odds)}")
print(f"Odds rows with a fight_id match: {matched_count} / {len(odds)}")
print(f"Odds rows with no match (fight_id = NaN): {odds['fight_id'].isna().sum()}")
print(f"r_is_f1 distribution:\n{odds['r_is_f1'].value_counts(dropna=False)}")

Dropped 1942 rows with no fight_id. Remaining rows with fight_id: 3586
Odds rows with a fight_id match: 3586 / 3586
Odds rows with no match (fight_id = NaN): 0
r_is_f1 distribution:
r_is_f1
1    3585
0       1
Name: count, dtype: Int64


In [163]:
# Merge on (pair, date)
odds_keys = odds[['pair', 'date_norm']].drop_duplicates()
stats_keys = fight_stats[['pair', 'date_norm']].drop_duplicates()

matched = odds_keys.merge(stats_keys, on=['pair', 'date_norm'], how='inner')

print(f'Odds rows:              {len(odds)}')
print(f'Fight stats rows:       {len(fight_stats)}')
print(f'Common fights (matched): {len(matched)}')
print(f'Odds with no match:      {len(odds) - len(odds.merge(matched, on=["pair","date_norm"], how="inner"))}')

Odds rows:              3586
Fight stats rows:       6900
Common fights (matched): 3586
Odds with no match:      0


In [164]:
# Coverage as % of each dataset
pct_odds = len(matched) / len(odds) * 100
pct_stats = len(matched) / len(fight_stats) * 100
print(f'{pct_odds:.1f}% of odds rows have a match in fight_stats')
print(f'{pct_stats:.1f}% of fight_stats rows have odds data')

100.0% of odds rows have a match in fight_stats
52.0% of fight_stats rows have odds data


In [165]:
# Sample unmatched odds rows to understand why they don't match
unmatched_odds = odds[~odds[['pair','date_norm']].apply(tuple, axis=1).isin(
    matched[['pair','date_norm']].apply(tuple, axis=1)
)]
print('Unmatched odds sample:')
# print(unmatched_odds[['R_fighter','B_fighter','date']].head(10))
print('\nUnmatched odds by year:')
pd.to_datetime(unmatched_odds['date_norm']).dt.year.value_counts().sort_index()

Unmatched odds sample:

Unmatched odds by year:


Series([], Name: count, dtype: int64)

## Clean Odds Data

In [166]:
print(odds.dtypes)
print('\nMissing values:')
print(odds.isnull().sum())

R_fighter        object
B_fighter        object
R_odds          float64
B_odds          float64
date             object
location         object
country          object
Winner           object
title_bout       object
weight_class     object
gender           object
r_norm           object
b_norm           object
pair             object
date_norm        object
fight_id          Int64
r_is_f1           Int64
dtype: object

Missing values:
R_fighter       0
B_fighter       0
R_odds          0
B_odds          0
date            0
location        0
country         0
Winner          0
title_bout      0
weight_class    0
gender          0
r_norm          0
b_norm          0
pair            0
date_norm       0
fight_id        0
r_is_f1         0
dtype: int64


In [167]:
print('Duplicate rows:', odds.duplicated().sum())
odds.drop_duplicates(inplace=True)

Duplicate rows: 0


In [168]:
# Drop rows missing odds or winner
before = len(odds)
odds.dropna(subset=['R_odds', 'B_odds', 'Winner'], inplace=True)
print(f'Dropped {before - len(odds)} rows with missing odds/winner. Remaining: {len(odds)}')

Dropped 0 rows with missing odds/winner. Remaining: 3586


In [169]:
# Parse date
odds['date'] = pd.to_datetime(odds['date'])

# Boolean flag columns
odds['title_bout'] = odds['title_bout'].astype(bool)

# Winner to binary: 1 = Red wins, 0 = Blue wins
odds['winner_red'] = (odds['Winner'].str.strip().str.lower() == 'red').astype(int)

print(odds.dtypes)

R_fighter               object
B_fighter               object
R_odds                 float64
B_odds                 float64
date            datetime64[ns]
location                object
country                 object
Winner                  object
title_bout                bool
weight_class            object
gender                  object
r_norm                  object
b_norm                  object
pair                    object
date_norm               object
fight_id                 Int64
r_is_f1                  Int64
winner_red               int64
dtype: object


In [170]:
# Verify odds are numeric and check range
print(odds[['R_odds','B_odds']].describe())

            R_odds       B_odds
count  3586.000000  3586.000000
mean   -116.441439    68.610987
std     267.158349   249.730168
min   -1650.000000 -1200.000000
25%    -255.000000  -145.000000
50%    -150.000000   130.000000
75%     130.000000   225.000000
max     775.000000  1300.000000


In [171]:
# Check weight class distribution
odds['weight_class'].value_counts()

weight_class
Lightweight          743
Welterweight         710
Middleweight         480
Featherweight        449
Bantamweight         388
Light Heavyweight    326
Heavyweight          295
Flyweight            182
Catch Weight          13
Name: count, dtype: int64

In [ ]:
# Convert US odds to implied probabilities
odds['r_prob'] = odds['R_odds'].astype(int).apply(lambda x: implied_prob(x)[0])
odds['b_prob'] = odds['B_odds'].astype(int).apply(lambda x: implied_prob(x)[0])

print('Implied probabilities added:')
print(odds[['R_odds', 'r_prob', 'B_odds', 'b_prob']].head(10))
print(f'\nProbability ranges:')
print(f'  r_prob: [{odds["r_prob"].min():.3f}, {odds["r_prob"].max():.3f}]')
print(f'  b_prob: [{odds["b_prob"].min():.3f}, {odds["b_prob"].max():.3f}]')

Implied probabilities added:
    R_odds    r_prob  B_odds    b_prob
0   -145.0  0.591837   135.0  0.425532
1    160.0  0.384615  -170.0  0.629630
2   -345.0  0.775281   315.0  0.240964
3   -137.0  0.578059   127.0  0.440529
4   -160.0  0.615385   150.0  0.400000
5   -150.0  0.600000   140.0  0.416667
10  -110.0  0.523810   100.0  0.500000
12  -260.0  0.722222   240.0  0.294118
14  -162.0  0.618321   152.0  0.396825
15   157.0  0.389105  -167.0  0.625468

Probability ranges:
  r_prob: [0.114, 0.943]
  b_prob: [0.071, 0.923]


In [173]:
# Drop normalization helper columns before saving
odds.drop(columns=['r_norm', 'b_norm', 'pair', 'date_norm'], inplace=True, errors='ignore')
print(odds.shape)
odds.head(3)

(3586, 16)


,R_fighter,B_fighter,R_odds,B_odds,date,location,country,Winner,title_bout,weight_class,gender,fight_id,r_is_f1,winner_red,r_prob,b_prob
0,Kevin Lee,Charles Oliveira,-145.0,135.0,2020-03-14,"Brasilia, Distrito Federal, Brazil",Brazil,Blue,False,Lightweight,MALE,1798,1,0,0.591837,0.425532
1,Demian Maia,Gilbert Burns,160.0,-170.0,2020-03-14,"Brasilia, Distrito Federal, Brazil",Brazil,Blue,False,Welterweight,MALE,1797,1,0,0.384615,0.629630
2,Renato Moicano,Damir Hadzovic,-345.0,315.0,2020-03-14,"Brasilia, Distrito Federal, Brazil",Brazil,Red,False,Lightweight,MALE,1792,1,1,0.775281,0.240964


## Save Cleaned Data

In [174]:
odds.to_csv('../data/processed/odds_cleaned.csv', index=False)
print('Saved to data/processed/odds_cleaned.csv with shape:', odds.shape)

Saved to data/processed/odds_cleaned.csv with shape: (3586, 16)
